In [49]:
!pip install pandas folium scikit-learn -q
import pandas as pd
import folium
from sklearn.linear_model import LogisticRegression
from IPython.display import HTML, display

# ======================
# 1. Tạo dữ liệu mẫu
# ======================
traffic = pd.DataFrame([
    {"road": "A", "lat": 10.761, "lon": 106.657, "volume": 300, "speed": 18, "accidents": 1},
    {"road": "B", "lat": 10.763, "lon": 106.659, "volume": 600, "speed": 8,  "accidents": 2},
    {"road": "C", "lat": 10.765, "lon": 106.662, "volume": 450, "speed": 12, "accidents": 1},
    {"road": "D", "lat": 10.759, "lon": 106.655, "volume": 700, "speed": 6,  "accidents": 3},
    {"road": "E", "lat": 10.760, "lon": 106.664, "volume": 280, "speed": 20, "accidents": 0},
])

# ======================
# 2. Tạo nhãn nguy cơ
# 1 = nguy cơ cao, 0 = nguy cơ thấp
# ======================
traffic["risk"] = (
    ((traffic["volume"] > 500) & (traffic["speed"] < 10)) |
    (traffic["accidents"] >= 2)
).astype(int)

print("Dữ liệu:")
print(traffic)

# ======================
# 3. Huấn luyện mô hình
# ======================
X = traffic[["volume", "speed", "accidents"]]
y = traffic["risk"]

model = LogisticRegression()
model.fit(X, y)

# Dự đoán
traffic["predicted_risk"] = model.predict(X)
traffic["probability"] = model.predict_proba(X)[:, 1]

print("\nKết quả dự đoán:")
print(traffic[["road", "predicted_risk", "probability"]])

# ======================
# 4. Vẽ bản đồ
# ======================
m = folium.Map(location=[10.762, 106.660], zoom_start=15)

for _, row in traffic.iterrows():
    if int(row["predicted_risk"]) == 1:
        color = "red"
    else:
        color = "green"

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=10,
        color=color,
        fill=True,
        fill_color=color,
        popup=f"Đường {row['road']}<br>Nguy cơ: {int(row['predicted_risk'])}<br>Xác suất: {row['probability']:.2f}"
    ).add_to(m)

display(HTML(m._repr_html_()))

Dữ liệu:
  road     lat      lon  volume  speed  accidents  risk
0    A  10.761  106.657     300     18          1     0
1    B  10.763  106.659     600      8          2     1
2    C  10.765  106.662     450     12          1     0
3    D  10.759  106.655     700      6          3     1
4    E  10.760  106.664     280     20          0     0

Kết quả dự đoán:
  road  predicted_risk   probability
0    A               0  2.761396e-10
1    B               1  9.993486e-01
2    C               0  6.519657e-04
3    D               1  1.000000e+00
4    E               0  3.887781e-11
